In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, f_oneway, chi2_contingency
from scipy.stats import ttest_ind

In [2]:
# importing all the cleaned file in python
viewing=pd.read_csv("viewing_activity_clean.csv")
ratings=pd.read_csv("ratings_feedback_clean.csv")
users=pd.read_csv("user_profile_clean.csv")
content=pd.read_csv("content_library_clean.csv")
subscription=pd.read_csv("subscription_retention_clean.csv")

In [3]:
viewing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   session_id              30000 non-null  object
 1   user_id                 30000 non-null  object
 2   content_id              30000 non-null  object
 3   view_date               30000 non-null  object
 4   watch_duration_minutes  30000 non-null  int64 
 5   completion_percentage   30000 non-null  int64 
 6   paused_times            30000 non-null  int64 
 7   rewatched_flag          30000 non-null  object
 8   time_of_day             30000 non-null  object
 9   device_type             30000 non-null  object
dtypes: int64(3), object(7)
memory usage: 2.3+ MB


In [4]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            12000 non-null  object
 1   content_id         12000 non-null  object
 2   rating             12000 non-null  int64 
 3   liked_flag         12000 non-null  object
 4   feedback_category  12000 non-null  object
dtypes: int64(1), object(4)
memory usage: 468.9+ KB


In [5]:
subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            2000 non-null   object
 1   subscription_type  2000 non-null   object
 2   monthly_fee        2000 non-null   int64 
 3   renewal_status     2000 non-null   object
 4   churn_flag         2000 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 78.3+ KB


In [6]:
viewing['user_id'] = viewing['user_id'].str.strip().str.upper()
ratings['user_id'] = ratings['user_id'].str.strip().str.upper()
subscription['user_id'] = subscription['user_id'].str.strip().str.upper()
viewing['content_id'] = viewing['content_id'].str.strip().str.upper()
ratings['content_id'] = ratings['content_id'].str.strip().str.upper()

In [7]:
ratings_avg = ratings.groupby('user_id')['rating'].mean().reset_index()

In [8]:
df = viewing.merge(ratings_avg, on='user_id', how='left')
df = df.merge(subscription, on='user_id', how='left')

In [9]:
df.head()

,session_id,user_id,content_id,view_date,watch_duration_minutes,completion_percentage,paused_times,rewatched_flag,time_of_day,device_type,rating,subscription_type,monthly_fee,renewal_status,churn_flag
0,S723607,U0001,C08386,2024-09-17,5,13,0,Y,Morning,Smart TV,4.4,Free Trial,0,Not Renewed,1
1,S221333,U0001,C03723,2022-09-04,84,14,2,N,Evening,Smart TV,4.4,Free Trial,0,Not Renewed,1
2,S853250,U0001,C15629,2022-05-02,74,98,0,N,Night,Smart TV,4.4,Free Trial,0,Not Renewed,1
3,S562018,U0001,C01926,2025-02-02,97,76,0,Y,Afternoon,Smart TV,4.4,Free Trial,0,Not Renewed,1
4,S283293,U0001,C04236,2022-05-03,461,76,3,N,Afternoon,Smart TV,4.4,Free Trial,0,Not Renewed,1


In [10]:
df.drop_duplicates(inplace=True)

In [11]:
df['rating'] = df['rating'].fillna(df['rating'].mean())

In [12]:
df.head()

,session_id,user_id,content_id,view_date,watch_duration_minutes,completion_percentage,paused_times,rewatched_flag,time_of_day,device_type,rating,subscription_type,monthly_fee,renewal_status,churn_flag
0,S723607,U0001,C08386,2024-09-17,5,13,0,Y,Morning,Smart TV,4.4,Free Trial,0,Not Renewed,1
1,S221333,U0001,C03723,2022-09-04,84,14,2,N,Evening,Smart TV,4.4,Free Trial,0,Not Renewed,1
2,S853250,U0001,C15629,2022-05-02,74,98,0,N,Night,Smart TV,4.4,Free Trial,0,Not Renewed,1
3,S562018,U0001,C01926,2025-02-02,97,76,0,Y,Afternoon,Smart TV,4.4,Free Trial,0,Not Renewed,1
4,S283293,U0001,C04236,2022-05-03,461,76,3,N,Afternoon,Smart TV,4.4,Free Trial,0,Not Renewed,1


In [13]:
df.shape

(30000, 15)

In [14]:
df.isnull().sum()

session_id                0
user_id                   0
content_id                0
view_date                 0
watch_duration_minutes    0
completion_percentage     0
paused_times              0
rewatched_flag            0
time_of_day               0
device_type               0
rating                    0
subscription_type         0
monthly_fee               0
renewal_status            0
churn_flag                0
dtype: int64

In [15]:
df.describe()

,watch_duration_minutes,completion_percentage,paused_times,rating,monthly_fee,churn_flag
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,180.997733,52.355900,1.494967,3.003460,296.040500,0.444000
std,412.239248,28.083027,1.124315,0.645952,245.131158,0.496862
min,5.000000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,29.000000,28.000000,0.000000,2.600000,0.000000,0.000000
50%,61.000000,53.000000,1.000000,3.000000,299.000000,0.000000
75%,116.250000,77.000000,3.000000,3.400000,599.000000,1.000000
max,12169.000000,100.000000,3.000000,5.000000,599.000000,1.000000


## T-test

## Question: Do churned users watch differently than non-churned users?

## H0: No difference in watching time
## HA: Difference exist in watch time

In [16]:
active_users = df[df['churn_flag'] == 0]['watch_duration_minutes']
churned_users = df[df['churn_flag'] == 1]['watch_duration_minutes']

In [17]:
t_stat, p_value = ttest_ind(active_users ,churned_users)

print("T-stat:", t_stat)
print("P-value:", p_value)

T-stat: -1.2778281613255003
P-value: 0.2013199106136922


In [18]:
len(active_users)

16680

In [19]:
len(churned_users)

13320

In [19]:
if p_value < 0.05:
    print("Significant difference Reject H0")
else:
    print("No significant difference Fail to reject H0")

No significant difference Fail to reject H0


## There’s no clear difference in watch time between churned and active users.This means watch time alone doesn’t really explain why users leave.

# Chi-square test

## Subscription Type vs Churn Flag

In [20]:
from scipy.stats import chi2_contingency

table = pd.crosstab(df['subscription_type'], df['churn_flag'])
chi2, p, dof, expected = chi2_contingency(table)

In [21]:
print("Chi-square:", chi2)
print("P-value:", p)

Chi-square: 5516.863190609802
P-value: 0.0


In [22]:
len(table)

3

In [27]:
if p < 0.05:
    print("→ Significant relationship")
else:
    print("→ No relationship")

→ Significant relationship


## this shows significant relation between subscription type and churn flag which means they affect each other.

## Device type vs Churn Flag

In [33]:
table2 = pd.crosstab(df['device_type'], df['churn_flag'])
chi2, p, dof, expected = chi2_contingency(table2)

In [34]:
print("Chi-square:", chi2)
print("P-value:", p)

Chi-square: 41.81815532661549
P-value: 4.384918566015906e-09


In [35]:
if p < 0.05:
    print("→ Significant relationship")
else:
    print("→ No relationship")

→ Significant relationship


# Time of day Vs Churn Flag

In [36]:
table3= pd.crosstab(df['time_of_day'], df['churn_flag'])
chi2, p, dof, expected = chi2_contingency(table3)

In [37]:
print("Chi-square:", chi2)
print("P-value:", p)

Chi-square: 4.881929038723248
P-value: 0.18064987335885394


In [38]:
if p < 0.05:
    print("→ Significant relationship")
else:
    print("→ No relationship")

→ No relationship


## there is no significant relationship between time of day and churn flag.

## Rewatched flag vs churn flag

In [40]:
table4 = pd.crosstab(df['rewatched_flag'], df['churn_flag'])
chi2, p, dof, expected = chi2_contingency(table4)

In [41]:
print("Chi-square:", chi2)
print("P-value:", p)

Chi-square: 0.27684837270658014
P-value: 0.5987743847539857


In [42]:
if p < 0.05:
    print("→ Significant relationship")
else:
    print("→ No relationship")

→ No relationship
